In [18]:
""" Import packages """

import numpy as np
import LoadSaveFunctions as lsf

In [19]:
# Load Dataset

data = lsf.load_excel_file()

print(data.head(5))

Selected file: /home/grominou/Documents/Git_Projects/LinearModels/test.xlsx
Excel file loaded successfully.
   d18Op  d18OpSD  d18Ow  d18OwSD
0   13.8      0.2   -7.3      0.3
1   12.9      0.5   -7.3      0.1
2   14.9      0.3   -5.6      0.1
3   14.7      0.1   -5.6      0.6
4   19.4      0.8   -1.8      0.7


In [27]:
""" Define Functions """

def york_regression(x, y, sigma_x, sigma_y, rho):
    """Performs York et al. (2004) regression method."""
    n = len(x)
    beta = np.sum((x - np.mean(x)) * (y - np.mean(y))) / np.sum((x - np.mean(x))**2)  # OLS initial slope
    b = beta  # Initial slope estimate
    x_bar, y_bar = np.mean(x), np.mean(y)
    
    # Small epsilon to prevent division errors
    epsilon = 1e-10
    sigma_x[sigma_x == 0] = epsilon
    sigma_y[sigma_y == 0] = epsilon
    
    max_iter = 100  # Maximum iterations
    iteration = 0
    
    while iteration < max_iter:
        W_x = 1 / sigma_x**2
        W_y = 1 / sigma_y**2
        alpha = (W_x * W_y) ** 0.5
        W = W_x * W_y / (W_x + b**2 * W_y - 2 * b * rho * alpha + epsilon)  # Prevent division by zero
        x_hat = np.sum(W * x) / np.sum(W)
        y_hat = np.sum(W * y) / np.sum(W)
        U = x - x_hat
        V = y - y_hat
        beta_new = np.sum(W * V * (U / sigma_x)) / np.sum(W * (U**2 / sigma_x**2))
        
        if np.abs(beta_new - b) < 1e-6:  # More relaxed convergence condition
            break
        b = beta_new
        iteration += 1
    
    if iteration == max_iter:
        print("Warning: York regression did not fully converge.")
    
    a = y_hat - b * x_hat  # Intercept
    
    # Compute uncertainties following York et al. (2004)
    S = np.sum(W * (y - a - b * x)**2)
    sigma_b = np.sqrt(S / (np.sum(W * (x - x_hat)**2)))
    sigma_a = np.sqrt(sigma_b**2 * np.sum(W * x**2) / np.sum(W))
    
    return a, b, sigma_a, sigma_b

def prediction_error(x, y, sigma_x, sigma_y, rho, x_new):
    """Calculates the prediction error for a new x value using York regression."""
    a, b, sigma_a, sigma_b = york_regression(x, y, sigma_x, sigma_y, rho)
    
    # Estimated y_new
    y_new = a + b * x_new
    
    # Error propagation
    sigma_y_new = np.sqrt(sigma_a**2 + (x_new**2 * sigma_b**2))
    
    return y_new, sigma_y_new

In [29]:

x = np.array(data['d18Op'])
y = np.array(data['d18Ow'])
sigma_x = np.array(data['d18OpSD'])
sigma_y = np.array(data['d18OwSD'])
rho = np.zeros_like(x)  # No correlation
epsilon = 1e-10  # Small value to prevent division by zero
sigma_x[sigma_x == 0] = epsilon
sigma_y[sigma_y == 0] = epsilon


a, b, sigma_a, sigma_b = york_regression(x, y, sigma_x, sigma_y, rho)
if a < 0:
    label_york = 'y={:1.3f}x{:1.3f}'.format(b, a)
else:
    label_york = 'y={:1.3f}x+{:1.3f}'.format(b, a)
print("York regression: " + label_york)

York regression: y=0.113x-7.062


In [23]:
# Example Data
x = np.array([1, 2, 3, 4, 5])
y = np.array([2.1, 2.9, 3.8, 5.1, 5.9])
sigma_x = np.array([0.1, 0.1, 0.1, 0.1, 0.1])
sigma_y = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
rho = np.zeros_like(x)  # No correlation

x_new = 5
y_pred, sigma_y_pred = prediction_error(x, y, sigma_x, sigma_y, rho, x_new)
a, b = york_regression(x, y, sigma_x, sigma_y, rho)
if a < 0:
    label_york = 'y={:1.3f}x{:1.3f}'.format(b, a)
else:
    label_york = 'y={:1.3f}x+{:1.3f}'.format(b, a)
print("York regression: " + label_york)
print(f"Predicted y: {y_pred:.3f} ± {sigma_y_pred:.3f}")

York regression: y=0.098x+3.666
Predicted y: 4.156 ± 0.200
